## Gold — `dim_municipio` (DTB)

**Origem:** `workspace.silver.municipios` → **Destino:** `workspace.gold.dim_municipio`

- **Modelo:** Star Schema. `dim_municipio` é dimensão conformada de território, pronta para joins com fatos por `codigo_municipio` (IBGE) **ou** `codigo_tom` (ex.: obras do CNO, que usa TOM).
- **Grão:** 1 linha por município (SCD tipo 1 — versão corrente da 2025).
- **Transformações:**
  - Valida novamente a Natural Key (`codigo_municipio` = `^[0-9]{7}$`).
  - Carrega `codigo_tom` (4 dígitos, SIAFI/Tesouro) já enriquecido na silver via biblioteca [`cidade-ibge-tom`](https://pypi.org/project/cidade-ibge-tom/) (MIT).
  - `sk_municipio` = surrogate key sequencial (`row_number` ordenado por `codigo_municipio`, determinística entre execuções).
  - Reordena colunas: SK primeiro, depois NK e atributos hierárquicos (município → UF → regiões).
- **Linhagem:** ODS IBGE → `bronze.dtb` → `silver.municipios` (+ enriquecimento TOM via cidade-ibge-tom/SIAFI) → `gold.dim_municipio`.

In [0]:
%run ./_setup_dtb

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from data_pipeline import save_table, add_column_comments
from metadata.metadata import DIM_MUNICIPIO_COMMENTS

In [0]:
SOURCE_TABLE = "workspace.silver.municipios"
TARGET_TABLE = "workspace.gold.dim_municipio"

# Ordem das colunas na dimensão: SK, NK (+ TOM), atributos
COLUNAS_ORDENADAS = [
    "sk_municipio",
    "codigo_municipio",
    "codigo_tom",
    "nome_municipio",
    "sigla_uf",
    "codigo_uf",
    "nome_uf",
    "codigo_regiao_geografica_intermediaria",
    "nome_regiao_geografica_intermediaria",
    "codigo_regiao_geografica_imediata",
    "nome_regiao_geografica_imediata",
]

In [0]:
df = spark.table(SOURCE_TABLE)
print(f"Silver in: {df.count():,} linhas | colunas: {df.columns}")

# Garante NK válida antes de gerar a SK
df = df.filter(F.col("codigo_municipio").rlike("^[0-9]{7}$"))

# Surrogate key determinística: ordenação pela natural key
w = Window.orderBy("codigo_municipio")
df = df.withColumn("sk_municipio", F.row_number().over(w).cast("int"))

# Reordena colunas conforme contrato da dimensão
colunas = [c for c in COLUNAS_ORDENADAS if c in df.columns]
colunas += [c for c in df.columns if c not in COLUNAS_ORDENADAS]
df = df.select(*colunas)

print(f"Gold: {df.count():,} municípios na dimensão")
display(df.limit(10))

In [0]:
save_table(df, TARGET_TABLE)
add_column_comments(
    spark,
    TARGET_TABLE,
    DIM_MUNICIPIO_COMMENTS
)
print(f"Tabela {TARGET_TABLE} persistida: {spark.table(TARGET_TABLE).count():,} linhas")

In [0]:
total = spark.table(TARGET_TABLE).count()
distintos_sk = spark.table(TARGET_TABLE).select("sk_municipio").distinct().count()
distintos_nk = spark.table(TARGET_TABLE).select("codigo_municipio").distinct().count()
sem_tom = spark.table(TARGET_TABLE).filter(F.col("codigo_tom").isNull()).count()
com_tom = total - sem_tom
pct_tom = round(100 * com_tom / total, 2) if total else 0.0
print(f"Total: {total:,} | SK distintos: {distintos_sk:,} | NK distintos: {distintos_nk:,}")
print(f"Cobertura codigo_tom: {com_tom:,} ({pct_tom}%) | Sem TOM: {sem_tom:,}")
assert total == distintos_sk == distintos_nk, "Quebra de unicidade SK/NK em dim_municipio"
display(spark.sql(f"SELECT codigo_municipio, codigo_tom, nome_municipio, sigla_uf FROM {TARGET_TABLE} ORDER BY nome_municipio LIMIT 10"))
display(spark.sql(f"SELECT sigla_uf, count(*) AS qtd_municipios FROM {TARGET_TABLE} GROUP BY sigla_uf ORDER BY sigla_uf"))
display(spark.sql(f"SELECT nome_regiao_geografica_intermediaria, count(*) AS qtd_municipios FROM {TARGET_TABLE} GROUP BY nome_regiao_geografica_intermediaria ORDER BY qtd_municipios DESC LIMIT 10"))
display(spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}"))